In [6]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, rank, row_number
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.window import Window

# 1. Inisialisasi SparkSession
spark = (
    SparkSession.builder.appName("Tugas5_2505060025_Ramudya Fuji Pratama")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. Membaca dataset transaksi dari HDFS
df_transaksi = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv")
)

# 3. Menambahkan kolom pendapatan (unit_terjual * harga_satuan)
df_transaksi = df_transaksi.withColumn(
    "pendapatan", col("unit_terjual") * col("harga_satuan")
)

# 4. Membuat DataFrame df_target dari dictionary
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("Data transaksi dari HDFS:")
df_transaksi.show(5)
print("Data target cabang:")
df_target.show()

Data transaksi dari HDFS:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

Data target cabang:


[Stage 16:>                                                         (0 + 1) / 1]

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [7]:
# Meringkas total pendapatan per kota
df_total_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan hitung pencapaian_persen
df_bagian_a = df_total_kota.join(df_target, on="kota", how="inner").withColumn(
    "pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100
)

# Urutkan dari pencapaian tertinggi
hasil_a = df_bagian_a.orderBy(col("pencapaian_persen").desc())
hasil_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [8]:
# Agregasi total pendapatan per kota dan kategori
df_kat_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Definisi Window spec: partisi berdasarkan kota, urutkan dari pendapatan tertinggi
window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Terapkan row_number() dan filter peringkat 1
hasil_b = (
    df_kat_kota.withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
    .orderBy("kota")
)

hasil_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [9]:
# Mendaftarkan temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL murni
hasil_c = spark.sql("""
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_c.show()

[Stage 28:>                                                         (0 + 1) / 1]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Berdasarkan hasil analisis data transaksi dan pencapaian target penjualan, cabang Purworejo dengan PIC Fitri menunjukkan kinerja paling baik. Cabang Purworejo berhasil membukukan total pendapatan sebesar Rp45.650.000,00 dari target sebesar Rp30.000.000,00, yang berarti mencapai persentase pencapaian tertinggi yaitu sebesar 152,17%. Kinerja unggul ini juga didukung oleh tingginya aktivitas penjualan dengan total 116 transaksi, serta ditopang oleh kategori Kesehatan & Kecantikan sebagai kategori terlaris yang menyumbang pendapatan sebesar Rp10.075.000,00 di kota tersebut.

Sebaliknya, cabang Semarang dengan PIC Sari merupakan cabang yang paling membutuhkan perhatian manajemen. Cabang Semarang mencatatkan persentase pencapaian target paling rendah, yaitu hanya sebesar 69,41% (pendapatan Rp38.175.000,00 dari target Rp55.000.000,00) dengan volume penjualan 93 transaksi. Selain Semarang, cabang Magelang juga perlu dipantau secara ketat karena hanya mencapai 70,33% dari targetnya dengan total pendapatan Rp31.650.000,00 dari target Rp45.000.000,00 dan jumlah transaksi terendah yaitu 86 transaksi. Oleh karena itu, manajemen disarankan melakukan evaluasi strategi pemasaran dan promosi pada cabang Semarang dan Magelang.

In [10]:
spark.stop()
print("SparkSession berhasil ditutup.")

SparkSession berhasil ditutup.
